# MMS-TTS Bangla (`facebook/mms-tts-ben`)

Same **VITS** architecture as English. Separate checkpoint + Bengali-character tokenizer.
`is_uroman` is **False** — pass Bengali script directly, no romanization.

Kaggle: Internet ON. If English already ran in this session, skip pip and start at the load cell.

In [ ]:
!pip install -q --upgrade transformers accelerate

In [ ]:
from transformers import VitsModel, VitsTokenizer, set_seed
from IPython.display import Audio, display
import torch, numpy as np
from scipy.io import wavfile

MODEL_ID = "facebook/mms-tts-ben"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = VitsTokenizer.from_pretrained(MODEL_ID)
model = VitsModel.from_pretrained(MODEL_ID).to(device).eval()

print("is_uroman:", tokenizer.is_uroman)
print("language:", getattr(tokenizer, "language", None))
print("sampling_rate:", model.config.sampling_rate)
print("params_M:", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))
print("modules:", [n for n, _ in model.named_children()])

text = "স্বাগতম। এটি বাংলা টেক্সট টু স্পিচ মডেল।"
inputs = tokenizer(text, return_tensors="pt")
print("input_ids length:", inputs["input_ids"].shape[-1])

set_seed(555)
with torch.no_grad():
    waveform = model(**{k: v.to(device) for k, v in inputs.items()}).waveform[0].cpu().numpy()

print(f"seconds: {waveform.size / model.config.sampling_rate:.3f}")
display(Audio(waveform, rate=model.config.sampling_rate))

pcm16 = np.clip(waveform * 32767, -32768, 32767).astype(np.int16)
wavfile.write("mms_tts_ben_sample.wav", model.config.sampling_rate, pcm16)
print("saved mms_tts_ben_sample.wav")